In [1]:
from vizdoom import DoomGame

import random

import time

import numpy as np

from gymnasium import Env

from gymnasium.spaces import Discrete, Box

import cv2

from stable_baselines3 import PPO

In [2]:
class VizDoomGym(Env):
    # Initalizing function
    def __init__(self, render=False):
        super().__init__()
        self.game = DoomGame()
        self.game.load_config('ViZDoom/scenarios/defend_the_center.cfg')

        # Render frame logic
        self.game.set_window_visible(render)

        self.game.init()

        self.observation_space = Box(low = 0, high = 255, shape = (100, 160, 1), dtype=np.uint8)

        self.action_space = Discrete(3)

    # What is done when taking a step
    def step(self, action):
        # Specify action then take step
        actions = np.identity(3)
        reward = self.game.make_action(actions[action], 4)

        if self.game.get_state():
            state = self.game.get_state()
            img = state.screen_buffer
            img = self.grayscale(img)
            ammo = state.game_variables[0]
            info = {'ammo': ammo}

        else:
            img = np.zeros(self.observation_space.shape)
            info = {}
        
        done = self.game.is_episode_finished()

        #obs, reward, terminated, truncated, info
        return img, reward, done, False, info

    # Define how to render the game or environment
    def render():
        pass
    
    # What happens when a new game is started
    def reset(self, seed = None, options = None):
        super().reset(seed=seed)
        self.game.new_episode()
        info = {}

        return self.grayscale(self.game.get_state().screen_buffer), info

    # Grayscale the game frame and resize it
    def grayscale(self, observation):
        gray = cv2.cvtColor(np.moveaxis(observation, 0, -1), cv2.COLOR_BGR2GRAY)
        resize = cv2.resize(gray, (160, 100), interpolation=cv2.INTER_CUBIC)
        state = np.reshape(resize, (100, 160, 1))
        return state

    # Will be called to close the game
    def close(self):
        self.game.close()

In [ ]:
model = PPO.load("train/train_defend/best_model_120000")

env = VizDoomGym(render = True)

In [ ]:
for episode in range(5):
    obs, _ = env.reset()
    done = False
    total_reward = 0
    while not done:
        action, _ = model.predict(obs)
        obs, reward, done, _, info = env.step(action)
        time.sleep(0.01)
        total_reward += reward

    print(f'total reward for episode {episode} is {total_reward}')
    time.sleep(2)

env.close()